In [1]:
import pandas as pd

df = pd.read_csv("/content/phishing_site_urls.csv")

In [2]:
df.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [3]:
print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Dataset shape: (549346, 2)

Column names:
['URL', 'Label']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 549346 entries, 0 to 549345
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype 
---  ------  --------------   ----- 
 0   URL     549346 non-null  object
 1   Label   549346 non-null  object
dtypes: object(2)
memory usage: 8.4+ MB


In [4]:
df['Label'].value_counts()

,count
Label,
good,392924
bad,156422


In [5]:
print(df.isnull().sum())

URL      0
Label    0
dtype: int64


In [6]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 42150


In [9]:
df.groupby('URL')['Label'].nunique().value_counts()
#checking the duplicate values are for same label or not

,count
Label,
1,507194


In [8]:
# Remove URLs that have conflicting labels
conflicting_urls = df.groupby('URL')['Label'].nunique()
conflicting_urls = conflicting_urls[conflicting_urls > 1].index

df = df[~df['URL'].isin(conflicting_urls)]

# Remove exact duplicate rows
df = df.drop_duplicates()

# Reset row numbering
df = df.reset_index(drop=True)

print("Cleaned dataset shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

Cleaned dataset shape: (507194, 2)
Remaining duplicates: 0


In [10]:
df['Label'].value_counts()
#check count of label(1.legitimate, 2.phishing)

,count
Label,
good,392896
bad,114298


In [11]:
df['Label'].value_counts(normalize=True) * 100

,proportion
Label,
good,77.464639
bad,22.535361


In [12]:
# Select 5,000 legitimate URLs
good_urls = df[df['Label'] == 'good'].sample(
    n=5000, random_state=42
)

# Select 5,000 phishing URLs
bad_urls = df[df['Label'] == 'bad'].sample(
    n=5000, random_state=42
)

# Combine both categories
df_sample = pd.concat([good_urls, bad_urls])

# Shuffle the dataset
df_sample = df_sample.sample(
    frac=1, random_state=42
).reset_index(drop=True)

print("Sample dataset shape:", df_sample.shape)
print(df_sample['Label'].value_counts())

Sample dataset shape: (10000, 2)
Label
bad     5000
good    5000
Name: count, dtype: int64


In [13]:
print("Dataset shape:", df_sample.shape)

print("\nLabel distribution:")
print(df_sample['Label'].value_counts())

print("\nFirst 5 rows:")
display(df_sample.head())

Dataset shape: (10000, 2)

Label distribution:
Label
bad     5000
good    5000
Name: count, dtype: int64

First 5 rows:


,URL,Label
0,rebasa.tkpittman.com/new.swz?stock=&amp;I=WNSB...,bad
1,www.shadow-warriors.co.uk/Shadow_Con3.htm,good
2,ph.linkedin.com/in/endoking,good
3,answers.yahoo.com/question/index?qid=200805110...,good
4,caps.fool.com/Ticker/ZHNE.aspx,good


In [14]:
df_sample.to_csv(
    "phishguard_dataset.csv",
    index=False
)

In [15]:
from google.colab import files
files.download("phishguard_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
url = "https://google.com"

print("URL:", url)
print("Length:", len(url))
print("Number of dots:", url.count("."))
print("Number of hyphens:", url.count("-"))
print("Number of digits:", sum(c.isdigit() for c in url))

URL: https://google.com
Length: 18
Number of dots: 1
Number of hyphens: 0
Number of digits: 0


In [17]:
def extract_features(url):
    return {
        'url_length': len(url),
        'dot_count': url.count('.'),
        'hyphen_count': url.count('-'),
        'https': int(url.startswith('https')),
        'digit_count': sum(c.isdigit() for c in url)
    }

In [18]:
extract_features("https://google.com")

{'url_length': 18,
 'dot_count': 1,
 'hyphen_count': 0,
 'https': 1,
 'digit_count': 0}

In [19]:
features = df_sample['URL'].apply(extract_features)

features_df = pd.DataFrame(features.tolist())

display(features_df.head())

,url_length,dot_count,hyphen_count,https,digit_count
0,171,3,2,0,6
1,41,4,1,0,1
2,27,2,0,0,0
3,58,2,0,0,16
4,30,3,0,0,0


In [20]:
df_final = features_df.copy()

df_final['Label'] = df_sample['Label'].values

display(df_final.head())

,url_length,dot_count,hyphen_count,https,digit_count,Label
0,171,3,2,0,6,bad
1,41,4,1,0,1,good
2,27,2,0,0,0,good
3,58,2,0,0,16,good
4,30,3,0,0,0,good


In [21]:
df_final['Label'] = df_final['Label'].map({
    'good': 0,
    'bad': 1
})

display(df_final.head())

,url_length,dot_count,hyphen_count,https,digit_count,Label
0,171,3,2,0,6,1
1,41,4,1,0,1,0
2,27,2,0,0,0,0
3,58,2,0,0,16,0
4,30,3,0,0,0,0


In [22]:
print(df_final['Label'].value_counts())


Label
1    5000
0    5000
Name: count, dtype: int64


In [23]:
from sklearn.model_selection import train_test_split

X = df_final.drop('Label', axis=1)
y = df_final['Label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [24]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (8000, 5)
Testing data: (2000, 5)


In [25]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=1000)

In [26]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [27]:
y_pred = model.predict(X_test)

print("Actual values:", y_test.values[:10])
print("Predicted values:", y_pred[:10])

Actual values: [0 1 0 1 0 0 0 1 0 1]
Predicted values: [1 1 1 1 0 1 0 1 0 0]


In [28]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")

Model Accuracy: 0.662
Accuracy Percentage: 66.2 %


In [29]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

Confusion Matrix:
[[768 232]
 [444 556]]


In [30]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate", "Phishing"]
))

              precision    recall  f1-score   support

  Legitimate       0.63      0.77      0.69      1000
    Phishing       0.71      0.56      0.62      1000

    accuracy                           0.66      2000
   macro avg       0.67      0.66      0.66      2000
weighted avg       0.67      0.66      0.66      2000



In [31]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [32]:
from sklearn.linear_model import LogisticRegression

model_scaled = LogisticRegression(max_iter=1000)

model_scaled.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [33]:
y_pred_scaled = model_scaled.predict(X_test_scaled)

print("Actual values:", y_test.values[:10])
print("New predictions:", y_pred_scaled[:10])


Actual values: [0 1 0 1 0 0 0 1 0 1]
New predictions: [1 1 1 1 0 1 0 1 0 0]


In [34]:
from sklearn.metrics import accuracy_score

old_accuracy = accuracy_score(y_test, y_pred)
new_accuracy = accuracy_score(y_test, y_pred_scaled)

print("Original Model Accuracy:", old_accuracy * 100, "%")
print("Scaled Model Accuracy:", new_accuracy * 100, "%")

Original Model Accuracy: 66.2 %
Scaled Model Accuracy: 66.2 %


In [35]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_scaled,
    target_names=["Legitimate", "Phishing"]
))

              precision    recall  f1-score   support

  Legitimate       0.63      0.77      0.69      1000
    Phishing       0.71      0.56      0.62      1000

    accuracy                           0.66      2000
   macro avg       0.67      0.66      0.66      2000
weighted avg       0.67      0.66      0.66      2000



In [36]:
import joblib

joblib.dump(model_scaled, 'phishguard_model.pkl')
joblib.dump(scaler, 'phishguard_scaler.pkl')

['phishguard_scaler.pkl']

In [39]:
test_url = "https://google.com"

new_features = extract_features(test_url)

new_df = pd.DataFrame([new_features])

new_df_scaled = scaler.transform(new_df)

prediction = model_scaled.predict(new_df_scaled)

if prediction[0] == 1:
    print("Potentially Phishing URL")
else:
    print("Likely Legitimate URL")

#false positive

Potentially Phishing URL


In [38]:
test_url = "http://example.com"

new_features = extract_features(test_url)
new_df = pd.DataFrame([new_features])
new_df_scaled = scaler.transform(new_df)

prediction = model_scaled.predict(new_df_scaled)

if prediction[0] == 1:
    print("Potentially Phishing URL")
else:
    print("Likely Legitimate URL")

Likely Legitimate URL
